# Ente umfahren — Analyse, Problem & Lösungsansatz (Avoid Duckie)

**Stand: 2026-06-18.** Dieses Notebook hält den *durchdachten* Stand fest, wie der Bot
zuverlässig und persistent um Enten herumfahren soll — speziell der schwere Fall:
**Ente in einer Kurve.** Es ist eine **Design-/Analyse-Notiz**, der Ansatz ist noch
**nicht implementiert**. Detektion/Freezes stehen separat in `duckie_detection.ipynb`.

Enthält nur **gesicherte** Schlüsse (am Code verifiziert bzw. von Felix bestätigt).
Was sich als Fehlannahme herausgestellt hat, steht bewusst unter §11, damit es nicht
nochmal versucht wird.


## 1. Aktueller Stand der Ausweichlogik (verifiziert)

- `detect_duckies_node` rechnet seine Ausweichlogik auf dem **rohen, flachen** 640×480-Bild.
- `crop_img` (perspektivische Entzerrung) ist im Enten-Node zwar **definiert, wird aber
  NICHT aufgerufen** → keine Entzerrung. Nur `detect_lane_node` nutzt `crop_img`.
- Ablauf heute: nächste Ente (`lowest_duckie`, größtes `y2`) → Ente maskieren → Linien
  suchen (gelb links / weiß rechts, mit Bildrand-Fallback) → **breiteste Lücke in EINER
  Zeile** (an der Ente) → auf deren Mitte lenken.
- Wichtig: Im Problemfall ist die Ente **eindeutig im `min`/`max`-Bereich** → sie wird
  erkannt und ist aktiv. **Kein** Timing-/Gating-Problem.


## 2. Das eigentliche Problem

> **Die breiteste freie Fläche liegt in der Kurve AUSSERHALB der Strecke** (Kurvenaußenseite /
> Boden neben der Bahn). Der Bot folgt ihr → verlässt die Kurve bzw. fährt in die Ente, weil
> seine Lückensuche **nicht weiß, wohin die Bahn geht**.

Dazu:
- **Follow-Lane ist dummes Punkt-Folgen**: ein Punkt, dahin lenken. Es hat **kein** Konzept
  „da kommt eine Kurve". Kurvenerkennung lässt sich also nicht dranflanschen — sie existiert
  dort nicht.
- Der Bot „richtet sich eher an der weißen Linie aus / fährt eher rechts" → in einer
  Linkskurve driftet er damit nach außen / in die Ente.
- Der **Links/rechts-Vergleich** der Lücken hat „ich fahre geradeaus" fest eingebaut → in der
  Kurve ist das die falsche Achse.
- **Es ist (soweit bekannt) immer eine LINKSKURVE.** → Kurven*richtung* muss nicht erkannt
  werden, nur dass „gleich eine Kurve kommt".


## 3. Was zuverlässig ist – und was nicht

**Verlässlich:**
- Enten-Bounding-Box (v.a. **horizontale Lage** + **Unterkante** `y2` als grobe Tiefe). YOLO
  läuft auf GPU, ~12 Hz, stabil.
- **Nahbereich** der Strecke (unteres Bildband).

**Nicht verlässlich:**
- **Fernes Weiß**: Hintergrund (Wand, Teppich) und Auflösung — weit weg unscharf/verrauscht.
- **Absolute Lückenbreite nach vorne**: perspektivisch auf ~1 Pixel gestaucht.
- **Exakter Enten-Fußabdruck**: 2D-Box eines 3D-Objekts (siehe §8).

→ Der Ansatz darf sich **nur** auf Enten-Lage + Nah-Strecke stützen.


## 4. Perspektive – die harten Fakten

- **Nur flache Frontsicht**, keine echte Draufsicht. (Die Vogelperspektive in „Richtiger
  Weg.png" war nur zur Veranschaulichung.)
- Eine **nach vorne** verlaufende Lücke (Ente↔Linie) ist im flachen Bild auf ~1 px gestaucht
  → **absolute Breite dort nicht messbar.**
- `crop_img` IST eine perspektivische Entzerrung (Trapez: oben ~282 px schmal, unten ~635 px
  breit), aber **per Augenmaß** gesetzt, **nicht eingemessen** → nicht metrisch verlässlich.
  Und sie wird nur in `detect_lane` benutzt.
- Eine *saubere* Homographie rektifiziert **beide** Richtungen im **Maßstab** (auch vorwärts).
  Der Haken ist die **Auflösung**: das Ferne (oben) wird aus wenigen Pixeln hochgestreckt →
  maßstäblich ok, aber unscharf. Strecken erzeugt **keine** Information.
- **Entscheidung: ohne Entzerrung weiterarbeiten** (Auflösungsverlust oben; handgetunt eh
  nicht metrisch). Statt Warp → Höhen-Faktor (siehe §7).


## 5. Lösungsansatz – EINE Korridor-Analyse, nicht drei Module

„Kommt eine Kurve? — Ist da eine Ente? — Wo fahren wir lang?" sind **eine** Analyse, wenn man
statt **einem Punkt** den **befahrbaren Korridor über die Bildhöhe** (nah → mittel) betrachtet:

- **Kurve** = Korridor-Mitte wandert nach oben hin zur Seite.
- **Ente** = Blockade im Korridor (an ihrer Bodenzeile, §8).
- **Wo lang** = der zusammenhängende freie Pfad durch den Korridor.

Tragende Vereinfachungen (von Felix):
- **„Es gibt IMMER einen Weg auf der Strecke, wo der Bot durchpasst."** → Passierbarkeit muss
  nicht *bewiesen*, nur der Pfad *gefunden* werden. (Entfernt das schwerste Teilproblem.)
- **Höhen-Faktor** statt Warp (§7) für fairen Lückenvergleich.
- **Immer Linkskurve** → Richtung bekannt.
- **An der Strecke/Kurve verankern**, nie eine Lücke wählen, die die Strecke verlässt.
- **Persistenz/Latch** (§9): einmal entscheiden, festhalten.

Das ist ein **kleiner lokaler Pfadplaner auf dem Bild** statt eines Punkt-Folgers.


## 6. Kurve erkennen (Idee, robust)

Nicht die ferne Linie *präzise verorten* (fragil), sondern ein **aggregates, relatives**
Merkmal messen: **wie viel Weiß / ob Gelb da ist — verglichen mit der Geraden-Referenz.**

- Aggregat + relativ = robust gegen Rauschen und gegen das 1-Pixel-/Auflösungsproblem.
- Da Richtung bekannt (links), muss der Trigger nur **„Kurve voraus"** liefern.

**Stolpersteine:**
1. **Hintergrund-Weiß** (Wand/Teppich) täuscht „viel Weiß" vor → **ROI aufs untere Band**.
2. **„Gelb weg" ist hier heikel**: In der Problem-Szene war gar kein Gelb. Wenn Gelb auch auf
   Geraden oft fehlt, taugt „Gelb verschwindet" nicht als Kurvensignal → dann nur das
   **Weiß-Merkmal** nutzen. → **OFFENE FRAGE (§10): ist auf Geraden zuverlässig Gelb da?**


In [ ]:
# Skizze Kurven-Trigger (Pseudocode, NICHT eingebaut)
# Idee: Weiss/Gelb nur im unteren ROI zaehlen/grob verorten und mit Geraden-Referenz vergleichen.
#
# roi = bild[roi_top:roi_bottom, :]            # unteres Band -> Hintergrund-Weiss vermeiden
# weiss_menge   = anteil_weisser_pixel(roi)
# weiss_lage    = schwerpunkt_x(weisse_pixel(roi))
# gelb_da       = anteil_gelber_pixel(roi) > eps     # auf dieser Strecke evtl. unbrauchbar
#
# kurve_voraus = (abs(weiss_menge - REF_WEISS_GERADE) > tol_menge) or \
#                (abs(weiss_lage  - REF_LAGE_GERADE)  > tol_lage)  or \
#                (REF_GELB_GERADE and not gelb_da)
# # Richtung ist bekannt: immer links.


## 7. Höhen-Faktor statt Entzerrung

Eine Lücke weiter **oben** im Bild hat weniger Pixel, steht aber für **mehr** reale Strecke.
Statt das Bild zu warpen (Detailverlust oben): die **gemessene Pixel-Lücke mit einem
höhenabhängigen Faktor gewichten** → grobe reale Lückengröße.

- Das ist im Kern „nur die horizontale cm-pro-Pixel-Skala **pro Bildzeile**" — genau der Teil,
  den man für den Lückenvergleich braucht.
- **Kein Warp, kein Strecken, kein Detailverlust** — man **gewichtet die Messung**, statt das
  Bild umzurechnen.
- Faktor einmal grob einmessen (eine Zeile ≙ wie viele cm). Mehr nicht.


In [ ]:
# Skizze Hoehen-Faktor (Pseudocode)
# reale_breite ~ pixel_breite * faktor(zeile)
# faktor steigt mit der Hoehe (weiter weg = mehr cm pro Pixel).
#
# def reale_luecke(pixel_breite, zeile):
#     return pixel_breite * faktor(zeile)      # faktor(zeile) einmal grob eingemessen
#
# # damit werden Luecken auf verschiedenen Hoehen FAIR vergleichbar,
# # ohne das Bild zu entzerren.


## 8. Enten korrekt im Korridor behandeln (3D!) — WICHTIG

Die Ente ist 3D. Malt man die **ganze** Bounding-Box als „blockiert" in den Korridor, blockiert
ihr **Kopf** (sitzt physisch höher → höhere Bildzeile → **weiter entfernte** Fahrbahn) eine
Fläche, die in Wahrheit **frei** ist. Folge: Der Bot hält die Straße **hinter** der Ente für
zu → plant nie einen Weg an ihr vorbei.

**Lösung — Ente auf ihre Bodenzeile zusammenfalten:**
- Ente **nur an Zeile ~`y2`** (Unterkante = Bodenkontakt) als blockiert eintragen, schmales Band.
- **Alle Zeilen darüber bleiben frei** → Korridor hinter der Ente offen → Vorbeifahren planbar.
- **Breite = Box-Breite `[x1,x2]`** = **konservative Überschätzung** (echte Basis ist schmaler,
  wie Felix sagt) → etwas mehr Abstand = sicher.
- Unterkante nur für **Tiefe**, Box-Breite als **bewusst konservativer** seitlicher Block —
  **kein** präziser Fußabdruck (aus 2D-Box nicht rekonstruierbar). „Es gibt immer einen Weg"
  macht konservativ unkritisch.

**OFFENE FRAGE (§10):** steht die Ente immer aufrecht (dann ist `y2` verlässlich) oder kippt/
liegt sie mal?


In [ ]:
# Skizze: Ente als Block NUR an der Bodenzeile (Pseudocode)
# fuer jede erkannte Ente (x1,y1,x2,y2):
#     basis_zeile = y2                          # Bodenkontakt = Tiefe
#     block_links, block_rechts = x1, x2        # konservativ (Box-Breite)
#     korridor[basis_zeile-band : basis_zeile, block_links:block_rechts] = BLOCKIERT
#     # Zeilen OBERHALB y2 bleiben FREI -> Weg hinter der Ente bleibt planbar


## 9. Persistenz & Seitenkollision (Weg 1)

- **Latch:** Die Seitenwahl wird **einmal** getroffen (sobald die Ente in den relevanten
  Bereich kommt) und über das Manöver **festgehalten** → kein Frame-zu-Frame-Flackern.
- **Seitenkollision** (Ente seitlich neben dem Bot, außerhalb des Sichtfelds): Ursache =
  Frontkamera + **kein räumliches Gedächtnis**. Gewählter Ansatz **„Weg 1"** (leichtgewichtig):
  zuletzt gesehene Seite merken und das **Lenken zu dieser Seite kurz sperren**, **zeitlich/
  streckenbegrenzt** (gegen Linienüberfahren). (Weg 2 = Odometrie-Hinderniskarte → verworfen,
  zu aufwendig.)
- Die bereits eingebaute **Detektions-Hysterese** (Erkennung ein paar Frames halten) stützt
  das, wenn die Ente kurz verschwindet.


## 10. Der eigentliche Knackpunkt + offene Fragen

**Der harte Teil ist NICHT die Kurven-/Pfadlogik** (die ist dann fast geschenkt), sondern:
**den „befahrbaren Korridor" pro Zeile zuverlässig aus diesem Bild zu extrahieren** —
alles weiß, Hintergrund spielt rein, fern unscharf.

**Offene Fragen, bevor gebaut wird:**
1. **Woran „befahrbar" festmachen?** Dunkler Asphalt (frei = dunkel zwischen hellen Rändern)
   / Linien (Ränder finden, Korridor dazwischen) / Kombination? Was ist auf der Strecke am
   stabilsten?
2. **Gelb auf Geraden zuverlässig vorhanden?** (entscheidet §6-Kurventrigger)
3. **Ente immer aufrecht?** (entscheidet `y2`-Verlässlichkeit, §8)
4. Höhen-Faktor (§7) einmal grob einmessen.
5. ROI-Grenzen (oben/unten) festlegen — Hintergrund-Weiß raus, genug Nahbereich rein.

**Architektur-Notiz:** Kurveninfo aus dem Lane-Node zu holen geht nicht sinnvoll (dort nur
Punkt-Folgen, kein Kurvenkonzept; zudem rechnet Lane im entzerrten Raum, Enten-Node im rohen).
→ Korridor/Kurve im Enten-Node selbst aus dem Nah-ROI bestimmen.


## 11. Verworfene Annahmen (NICHT der Weg)

Bewusst dokumentiert, damit es nicht wiederholt wird:
- **„Nächste weiße Linie links der Mitte = linke Korridorgrenze."** Verworfen: Auf der Strecke
  ist **alles weiß** (beide Ränder + Hintergrund) → unzuverlässig. (Code-Versuch wurde
  zurückgerollt.)
- **`distance_min`/`max`-Gating war die Ursache.** Falsch — die Ente ist im Bereich.
- **Box-Unterkante = sauberer Fußabdruck.** Falsch — 2D-Box eines 3D-Objekts; korrekt ist
  §8 (nur Bodenzeile, konservative Breite).
- **Volle Entzerrung als Lösung.** Verworfen — Auflösungsverlust oben, handgetunt nicht
  metrisch; stattdessen Höhen-Faktor (§7).
- **„Geringste seitliche Abweichung" bei der Lückenwahl.** Verworfen — es soll IMMER die
  größte/auf der Strecke liegende Lücke sein.


## 12. Bereits umgesetzte, behaltene Fixes (Kontext)

Diese laufen schon (uncommitted, Branch `feature/duckie-detection-fixes`) und sind unabhängig
vom obigen Korridor-Ansatz:
- **Detektions-Hysterese** gegen Aussetzer (Maskierung + kurzer Nachlauf). Params:
  `detection.hold_frames`, `detection.mask_padding`.
- **B — Reinkriechen:** oberes Distanz-Gate gefixt (nah = max Gefahr) + **harter Stopp** sehr
  nah (`pid_duckie.stop_factor`).
- **A — Kurvenspeed:** Drossel-Untergrenze tunebar (`pid.min_speed_factor`).
- **Mehr-Enten-Lücke:** breiteste Lücke über alle Enten (einzeilig, Rohbild) — wird vom
  Korridor-Ansatz oben perspektivisch noch ersetzt/erweitert.

Siehe auch `duckie_detection.ipynb` (Detektion, Freezes, GPU).
